# Phase 17 — Data Analysis with DuckDB SQL

The purpose of this phase is to ask business questions and use SQL to produce the answer.

We will join `fact_sales` to `dim_date`, `dim_product`, `dim_store`, and `dim_customer`.

**No machine learning or AI is used in this phase.**

## Warehouse files required

- `data/fact_sales.csv`
- `data/dim_date.csv`
- `data/dim_product.csv`
- `data/dim_store.csv`
- `data/dim_customer.csv`

These are copied into this phase so the analysis notebook can run independently.

In [ ]:
import duckdb
con = duckdb.connect()
for t in ["dim_date","dim_customer","dim_product","dim_store","fact_sales"]:
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv_auto('data/{t}.csv')")


## Create the joined analysis view

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW sales_analysis AS
SELECT f.*, d.date, d.year, d.month, d.month_name, d.weekday, d.quarter,
       p.product_id, p.product_name, p.category,
       s.store_id, s.store_name, s.city AS store_city,
       c.customer_id, c.customer_name, c.segment, c.city AS customer_city
FROM fact_sales f
JOIN dim_date d ON f.date_id=d.date_id
JOIN dim_product p ON f.product_key=p.product_key
JOIN dim_store s ON f.store_key=s.store_key
JOIN dim_customer c ON f.customer_key=c.customer_key
""")


# Business Questions

The following are examples of the exact type of questions students should answer with SQL. The complete query bank is in `duckdb_business_questions.sql`.

### 1. When was the highest sales day?

In [ ]:
con.execute("""
SELECT date, SUM(sales_amount) AS total_sales
FROM sales_analysis
GROUP BY date
ORDER BY total_sales DESC
LIMIT 1
""").df()

### 2. When was the lowest sales day?

In [ ]:
con.execute("""
SELECT date, SUM(sales_amount) AS total_sales
FROM sales_analysis
GROUP BY date
ORDER BY total_sales ASC
LIMIT 1
""").df()

### 3. When were orders highest?

In [ ]:
con.execute("""
SELECT date, COUNT(DISTINCT transaction_id) AS orders
FROM sales_analysis
GROUP BY date
ORDER BY orders DESC, date
LIMIT 1
""").df()

### 4. Which product was best-selling on a particular date?

In [ ]:
con.execute("""
SELECT date, product_name, SUM(quantity) AS units, SUM(sales_amount) AS sales
FROM sales_analysis
WHERE date = DATE '2026-06-15'
GROUP BY date, product_name
ORDER BY units DESC, sales DESC
""").df()

### 5. Which product was best-selling in each week?

In [ ]:
con.execute("""
WITH weekly AS (
  SELECT year, EXTRACT('week' FROM date) AS week_number, product_name,
         SUM(quantity) AS units, SUM(sales_amount) AS sales
  FROM sales_analysis
  GROUP BY year, week_number, product_name
)
SELECT *
FROM weekly
QUALIFY ROW_NUMBER() OVER(
  PARTITION BY year, week_number ORDER BY units DESC, sales DESC
)=1
ORDER BY year, week_number
""").df()

### 6. When was each product's sales highest?

In [ ]:
con.execute("""
WITH daily_product AS (
  SELECT product_name, date, SUM(sales_amount) AS sales
  FROM sales_analysis
  GROUP BY product_name, date
)
SELECT *
FROM daily_product
QUALIFY ROW_NUMBER() OVER(PARTITION BY product_name ORDER BY sales DESC, date)=1
ORDER BY product_name
""").df()

### 7. When did each product last have an order?

In [ ]:
con.execute("""
SELECT product_name, MAX(date) AS last_order_date
FROM sales_analysis
GROUP BY product_name
ORDER BY last_order_date
""").df()

### 8. When did daily sales first fall significantly?

In [ ]:
con.execute("""
WITH daily AS (
  SELECT date, SUM(sales_amount) AS sales
  FROM sales_analysis GROUP BY date
), x AS (
  SELECT *, LAG(sales) OVER(ORDER BY date) AS previous_sales FROM daily
)
SELECT date, sales, previous_sales,
       ROUND(100*(sales-previous_sales)/NULLIF(previous_sales,0),2) AS pct_change
FROM x
WHERE previous_sales IS NOT NULL AND sales < previous_sales * 0.80
ORDER BY date
LIMIT 1
""").df()

### 9. When did a selected product have zero orders?

A simple fact query cannot find zero activity because zero means **no fact row exists**. We therefore create a complete date × product grid and left join sales.

In [ ]:
con.execute("""
WITH grid AS (
  SELECT d.date, p.product_key, p.product_name
  FROM dim_date d CROSS JOIN dim_product p
), sales AS (
  SELECT date_id, product_key, SUM(quantity) AS units
  FROM fact_sales GROUP BY date_id, product_key
)
SELECT g.date, g.product_name, COALESCE(s.units,0) AS units
FROM grid g
LEFT JOIN sales s
  ON s.date_id = CAST(STRFTIME(g.date,'%Y%m%d') AS INTEGER)
 AND s.product_key = g.product_key
WHERE COALESCE(s.units,0)=0
ORDER BY g.date, g.product_name
""").df()

## Required analysis method

For every business question, students should explicitly identify:

1. What does the question mean?
2. Which table contains the required measure?
3. Which dimension contains the filtering/grouping attribute?
4. What join is required?
5. What aggregation is required?
6. Does the question require a window function?
7. Does the question require a date × entity grid to detect zero activity?
8. What does the result mean for the business?

## More questions

Open `BUSINESS_QUESTION_BANK.md` and `duckdb_business_questions.sql`. The query bank includes more than 30 practical questions around dates, weeks, products, stores, customers, promotions, zero activity, trends, and significant decreases.